# Structural Damage Classification — MobileNetV2 Feature Extraction

This notebook replaces the HOG feature pipeline with the **same type of approach used by the uploaded reference notebook**:

**PEER Φ-Net images → ImageNet-pretrained MobileNetV2 → 1280-D CNN features**

The pretrained MobileNetV2 is used only as a **frozen feature extractor**. No classifier head is trained here.


## 1. Imports and configuration


In [2]:
import torch
import torchvision

print("Torch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.14.1+cpu
Torchvision version: 0.29.1+cpu
CUDA available: False


In [3]:
from pathlib import Path
import gc
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split

SEED = 42
VAL_FRACTION = 0.15

# Windows-friendly settings
NUM_WORKERS = 0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64 if torch.cuda.is_available() else 16

PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
print("Batch size:", BATCH_SIZE)
print("Configuration loaded.")


Project root: c:\Users\Noothan\Documents\my_work\structural-damage-classification
Device: cpu
Batch size: 16
Configuration loaded.


## 2. Dataset paths and file checks


In [7]:
# ============================================================
# 2. DATASET PATHS
# ============================================================

TASK_DIR_1 = DATA_DIR / "task2_damage_state_1"
TASK_DIR_2 = DATA_DIR / "task2_damage_state_2"

# Training images are in Task 2_2 folder
X_TRAIN_PATH = TASK_DIR_2 / "task2_X_train.npy"

# Training labels and test files are in Task 2_1 folder
Y_TRAIN_PATH = TASK_DIR_1 / "task2_y_train.npy"
X_TEST_PATH = TASK_DIR_1 / "task2_X_test.npy"
Y_TEST_PATH = TASK_DIR_1 / "task2_y_test.npy"

required_files = [
    X_TRAIN_PATH,
    Y_TRAIN_PATH,
    X_TEST_PATH,
    Y_TEST_PATH
]

missing = [str(p) for p in required_files if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Missing required files:\n" + "\n".join(missing)
    )

print("✓ All required dataset files are present.")
print()
print("X_train :", X_TRAIN_PATH)
print("y_train :", Y_TRAIN_PATH)
print("X_test  :", X_TEST_PATH)
print("y_test  :", Y_TEST_PATH)

✓ All required dataset files are present.

X_train : c:\Users\Noothan\Documents\my_work\structural-damage-classification\data\task2_damage_state_2\task2_X_train.npy
y_train : c:\Users\Noothan\Documents\my_work\structural-damage-classification\data\task2_damage_state_1\task2_y_train.npy
X_test  : c:\Users\Noothan\Documents\my_work\structural-damage-classification\data\task2_damage_state_1\task2_X_test.npy
y_test  : c:\Users\Noothan\Documents\my_work\structural-damage-classification\data\task2_damage_state_1\task2_y_test.npy


In [8]:
# ============================================================
# 3. LOAD DATA
# ============================================================

X_train_raw = np.load(X_TRAIN_PATH, mmap_mode="r")
y_train_raw = np.load(Y_TRAIN_PATH)

X_test_raw = np.load(X_TEST_PATH, mmap_mode="r")
y_test_raw = np.load(Y_TEST_PATH)

# Convert one-hot labels to class numbers
y_train_all = (
    np.argmax(y_train_raw, axis=1)
    if y_train_raw.ndim > 1
    else y_train_raw
)

y_test = (
    np.argmax(y_test_raw, axis=1)
    if y_test_raw.ndim > 1
    else y_test_raw
)

print("X_train:", X_train_raw.shape)
print("y_train:", y_train_all.shape)
print("X_test :", X_test_raw.shape)
print("y_test :", y_test.shape)

print("\nClass distribution:")
print("Train:", np.bincount(y_train_all))
print("Test :", np.bincount(y_test))

X_train: (11811, 224, 224, 3)
y_train: (11811,)
X_test : (1460, 224, 224, 3)
y_test : (1460,)

Class distribution:
Train: [6282 5529]
Test : [745 715]


In [9]:
# ============================================================
# 4. TRAIN / VALIDATION SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

indices = np.arange(len(y_train_all))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.15,
    stratify=y_train_all,
    random_state=42
)

y_train = y_train_all[train_idx]
y_val = y_train_all[val_idx]

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(y_test))

print("\nTrain classes:", np.bincount(y_train))
print("Validation classes:", np.bincount(y_val))
print("Test classes:", np.bincount(y_test))

Train: 10039
Validation: 1772
Test: 1460

Train classes: [5340 4699]
Validation classes: [942 830]
Test classes: [745 715]


## 3. Load labels and create the reproducible train/validation split

The official test set remains untouched. Only the original training pool is split into train and validation.


In [10]:
y_train_raw = np.load(Y_TRAIN_PATH)
y_test_raw = np.load(Y_TEST_PATH)

y_train_all = np.argmax(y_train_raw, axis=1) if y_train_raw.ndim > 1 else y_train_raw
y_test = np.argmax(y_test_raw, axis=1) if y_test_raw.ndim > 1 else y_test_raw

all_indices = np.arange(len(y_train_all))

train_idx, val_idx = train_test_split(
    all_indices,
    test_size=VAL_FRACTION,
    random_state=SEED,
    stratify=y_train_all
)

y_train = y_train_all[train_idx]
y_val = y_train_all[val_idx]

print("Training pool:", len(y_train_all))
print("Train:", len(y_train))
print("Validation:", len(y_val))
print("Test:", len(y_test))


Training pool: 11811
Train: 10039
Validation: 1772
Test: 1460


## 4. Memory-safe image dataset


In [11]:
X_train_raw = np.load(X_TRAIN_PATH, mmap_mode="r")
X_test_raw = np.load(X_TEST_PATH, mmap_mode="r")

print("X_train_raw:", X_train_raw.shape, X_train_raw.dtype)
print("X_test_raw :", X_test_raw.shape, X_test_raw.dtype)

# MobileNetV2 ImageNet preprocessing, matching the reference approach.
mobilenet_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class NumpyImageDataset(Dataset):
    def __init__(self, X):
        self.X = X

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        image = self.X[idx]
        if image.dtype != np.uint8:
            image = np.clip(image, 0, 255).astype(np.uint8)
        return mobilenet_transform(image)

train_pool_ds = NumpyImageDataset(X_train_raw)
test_ds = NumpyImageDataset(X_test_raw)

print("✓ Dataset and MobileNetV2 transform ready.")


X_train_raw: (11811, 224, 224, 3) float32
X_test_raw : (1460, 224, 224, 3) float32
✓ Dataset and MobileNetV2 transform ready.


## 5. Load ImageNet-pretrained MobileNetV2

The classifier layer is removed so the network outputs the **1280-dimensional feature representation**.


In [12]:
weights = models.MobileNet_V2_Weights.IMAGENET1K_V1
mobilenet = models.mobilenet_v2(weights=weights)

# Remove the final ImageNet classifier.
mobilenet.classifier = nn.Identity()
mobilenet.eval()
mobilenet.to(DEVICE)

print("✓ MobileNetV2 loaded.")
print("✓ Final classifier removed; CNN feature extraction mode enabled.")


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to C:\Users\Noothan/.cache\torch\hub\checkpoints\mobilenet_v2-b0353104.pth


100.0%


✓ MobileNetV2 loaded.
✓ Final classifier removed; CNN feature extraction mode enabled.


## 6. Extract and cache CNN features

This cell extracts features only once. If the cached feature files already exist, it loads them instead of repeating the extraction.


In [13]:
TRAIN_POOL_FEATURES = RESULTS_DIR / "mobilenetv2_train_pool.npy"
TEST_FEATURES = RESULTS_DIR / "mobilenetv2_test.npy"

def extract_features_to_memmap(model, dataset, output_path, batch_size):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )

    # Infer feature dimension from one batch.
    with torch.no_grad():
        first_batch = next(iter(loader)).to(DEVICE, non_blocking=True)
        first_features = model(first_batch).flatten(1)
        feature_dim = first_features.shape[1]

    features = np.lib.format.open_memmap(
        output_path,
        mode="w+",
        dtype="float32",
        shape=(len(dataset), feature_dim)
    )

    with torch.no_grad():
        start = 0
        for batch_num, batch in enumerate(loader, start=1):
            batch = batch.to(DEVICE, non_blocking=True)
            batch_features = model(batch).flatten(1).cpu().numpy().astype(np.float32)

            end = start + len(batch_features)
            features[start:end] = batch_features
            start = end

            if batch_num % 20 == 0 or end == len(dataset):
                print(f"Processed {end}/{len(dataset)} images")

    features.flush()
    return features.shape

if TRAIN_POOL_FEATURES.exists() and TEST_FEATURES.exists():
    X_train_pool_features = np.load(TRAIN_POOL_FEATURES, mmap_mode="r")
    X_test_features = np.load(TEST_FEATURES, mmap_mode="r")
    print("✓ Existing MobileNetV2 features loaded from results/.")
else:
    start_time = time.time()

    train_shape = extract_features_to_memmap(
        mobilenet, train_pool_ds, TRAIN_POOL_FEATURES, BATCH_SIZE
    )
    test_shape = extract_features_to_memmap(
        mobilenet, test_ds, TEST_FEATURES, BATCH_SIZE
    )

    X_train_pool_features = np.load(TRAIN_POOL_FEATURES, mmap_mode="r")
    X_test_features = np.load(TEST_FEATURES, mmap_mode="r")

    print(f"✓ Feature extraction completed in {(time.time() - start_time)/60:.2f} minutes.")

print("Training-pool features:", X_train_pool_features.shape)
print("Test features:", X_test_features.shape)


Processed 320/11811 images
Processed 640/11811 images
Processed 960/11811 images
Processed 1280/11811 images
Processed 1600/11811 images
Processed 1920/11811 images
Processed 2240/11811 images
Processed 2560/11811 images
Processed 2880/11811 images
Processed 3200/11811 images
Processed 3520/11811 images
Processed 3840/11811 images
Processed 4160/11811 images
Processed 4480/11811 images
Processed 4800/11811 images
Processed 5120/11811 images
Processed 5440/11811 images
Processed 5760/11811 images
Processed 6080/11811 images
Processed 6400/11811 images
Processed 6720/11811 images
Processed 7040/11811 images
Processed 7360/11811 images
Processed 7680/11811 images
Processed 8000/11811 images
Processed 8320/11811 images
Processed 8640/11811 images
Processed 8960/11811 images
Processed 9280/11811 images
Processed 9600/11811 images
Processed 9920/11811 images
Processed 10240/11811 images
Processed 10560/11811 images
Processed 10880/11811 images
Processed 11200/11811 images
Processed 11520/118

## 7. Create train/validation feature files


In [14]:
X_train_features = np.asarray(X_train_pool_features[train_idx], dtype=np.float32)
X_val_features = np.asarray(X_train_pool_features[val_idx], dtype=np.float32)
X_test_features = np.asarray(X_test_features, dtype=np.float32)

TRAIN_FEATURES = RESULTS_DIR / "X_train_mobilenetv2.npy"
VAL_FEATURES = RESULTS_DIR / "X_val_mobilenetv2.npy"
TEST_FEATURES_FINAL = RESULTS_DIR / "X_test_mobilenetv2.npy"

np.save(TRAIN_FEATURES, X_train_features)
np.save(VAL_FEATURES, X_val_features)
np.save(TEST_FEATURES_FINAL, X_test_features)

print("Saved:")
print(" ", TRAIN_FEATURES)
print(" ", VAL_FEATURES)
print(" ", TEST_FEATURES_FINAL)
print()
print("Train:", X_train_features.shape)
print("Validation:", X_val_features.shape)
print("Test:", X_test_features.shape)


Saved:
  c:\Users\Noothan\Documents\my_work\structural-damage-classification\results\X_train_mobilenetv2.npy
  c:\Users\Noothan\Documents\my_work\structural-damage-classification\results\X_val_mobilenetv2.npy
  c:\Users\Noothan\Documents\my_work\structural-damage-classification\results\X_test_mobilenetv2.npy

Train: (10039, 1280)
Validation: (1772, 1280)
Test: (1460, 1280)


## 8. Feature and label checks


In [15]:
assert X_train_features.shape[0] == len(y_train)
assert X_val_features.shape[0] == len(y_val)
assert X_test_features.shape[0] == len(y_test)

assert X_train_features.shape[1] == 1280
assert np.isfinite(X_train_features).all()
assert np.isfinite(X_val_features).all()
assert np.isfinite(X_test_features).all()

assert set(train_idx).isdisjoint(set(val_idx))

print("✓ Feature/label alignment verified.")
print("✓ Train/validation indices are disjoint.")
print("✓ MobileNetV2 feature dimension:", X_train_features.shape[1])


✓ Feature/label alignment verified.
✓ Train/validation indices are disjoint.
✓ MobileNetV2 feature dimension: 1280


## 9. Final summary

The project now uses a **frozen ImageNet-pretrained MobileNetV2 feature extractor** followed by classical machine-learning models. Logistic Regression and SVM will use the **same 1280-dimensional features**, the same train/validation/test split, and training-only scaling.


In [16]:
print("Preprocessing complete.")
print("Feature pipeline: Image → MobileNetV2 → 1280-D features")
print("Models ready: Logistic Regression and SVM")
print("Test set remained untouched during model training.")


Preprocessing complete.
Feature pipeline: Image → MobileNetV2 → 1280-D features
Models ready: Logistic Regression and SVM
Test set remained untouched during model training.
